# Baseline Evaluation

Runs physics and learned baselines against the IQ L=50 dataset.
All baselines are compared on the same held-out test split.

**Metrics** (accumulated globally across the whole test set, not averaged per-batch):
- MSE in log1p(I) space
- R² in log1p(I) space
- CPU time per atom (μs), atom-count-weighted

**Checkpointing / resume**: results are pushed to Drive (via the same `RCLONE_CONF`
Kaggle secret used by `kaggle_train.ipynb`) after every baseline finishes. If the
session dies mid-run, just rerun the notebook top to bottom -- already-completed
baselines are detected and skipped automatically.

In [ ]:
import os, subprocess, sys

NOTEBOOK_NAME = "your-kaggle-notebook"   # set to your Kaggle notebook slug
assert NOTEBOOK_NAME != "your-kaggle-notebook", "Set NOTEBOOK_NAME first."

HDF5_PATH = f"/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5"
REPO      = f"/kaggle/working/{NOTEBOOK_NAME}"
DB_NAME   = f"{REPO}/Preprocess/iq_train_set"   # reuses the LFS-tracked iq_train_set-ENCODING.sqlite3
N_BUCKETS = 10   # number of atom-size buckets to evaluate, randomly sampled (< 57 for speed)

# install deps
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py scikit-learn
!apt-get install -y -q git-lfs
!git lfs install
!curl -fsSL https://rclone.org/install.sh | sudo bash

# clone repo
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/noshou/APS360.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

# force-fetch real LFS content (a clone/pull before git-lfs was installed above
# would have left LFS files as small pointer stubs instead of the real blobs)
subprocess.run(["git", "-C", REPO, "lfs", "pull"], check=True)

sys.path.insert(0, REPO)

In [ ]:
# ── rclone / Google Drive checkpointing ── run once per session ─────────────
# Reuses the same RCLONE_CONF Kaggle secret as kaggle_train.ipynb (it's just
# remote storage credentials, not training-specific). Checkpoints under a
# separate baselines_ckpts/ prefix so this never collides with training
# checkpoints on the same Drive.
import base64, json
from kaggle_secrets import UserSecretsClient

conf_path = "/kaggle/working/rclone.conf"
with open(conf_path, "w") as f:
    f.write(base64.b64decode(UserSecretsClient().get_secret("RCLONE_CONF")).decode())
os.environ["RCLONE_CONFIG"] = conf_path

remotes = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True).stdout.strip().split("\n")
remote  = remotes[0] if remotes and remotes[0] else ""
assert remote.endswith(":"), (
    f"No rclone remote found (rclone listremotes -> {remotes!r}). Check the RCLONE_CONF secret."
)
REMOTE_NAME = remote + f"{NOTEBOOK_NAME}/baselines_ckpts/"

out = subprocess.run(["rclone", "mkdir", REMOTE_NAME], capture_output=True, text=True)
print("remote drive ──>", REMOTE_NAME, "(ok)" if out.returncode == 0 else f"ERROR: {out.stderr.strip()}")

_CKPT_LOCAL = "/kaggle/working/baselines_results.json"

def load_checkpoint() -> dict:
    """Pull results.json from Drive if it exists and return completed baselines.

    Returns an empty dict on a fresh run (nothing to resume). Baselines whose
    name is already a key here get skipped by the evaluation loops below.
    """
    subprocess.run(["rclone", "copy", f"{REMOTE_NAME}results.json", "/kaggle/working/"],
                    capture_output=True)
    if not os.path.exists(_CKPT_LOCAL):
        return {}
    with open(_CKPT_LOCAL) as f:
        data = json.load(f)
    data = {k: tuple(v) for k, v in data.items()}
    print(f"Resumed {len(data)} completed baseline(s) from checkpoint: {list(data.keys())}")
    return data

def save_checkpoint(results: dict) -> None:
    """Write results.json locally and push it to Drive. Call after every baseline."""
    with open(_CKPT_LOCAL, "w") as f:
        json.dump(results, f, indent=2)
    subprocess.run(["rclone", "copy", _CKPT_LOCAL, REMOTE_NAME], capture_output=True)

In [ ]:
import h5py, hdf5plugin
from Preprocess.encode import Encoding
from ScatterNet.utils.config import DEFAULT_BUCKETS

print("Loading encoding DB (reuses iq_train_set-ENCODING.sqlite3 from git-lfs)...")
enc = Encoding(DB_NAME, HDF5_PATH)
print(f"  {enc.count():,} molecules  |  max atoms: {enc._max}")

with h5py.File(HDF5_PATH, "r") as f:
    q_grid = f["q_grid"][()]
    energy = float(f.attrs.get("energy", 10000.0))

import torch
q_grid = torch.from_numpy(q_grid).float()
print(f"  q_grid: {len(q_grid)} points  |  energy: {energy} eV")

In [ ]:
import random
import time
from ScatterNet.batching import Batcher, Batch
from torch.utils.data import DataLoader

BUCKET_SAMPLE_SEED = 3092983   # deterministic; change to sample a different subset of buckets
LOADER_WORKERS = 4              # parallel HDF5 reads while materializing each split once

def _first(x):
    return x[0]

def _materialize(dataset, name, num_workers=LOADER_WORKERS):
    """Read every batch out of `dataset` once via a parallel DataLoader and cache
    it as a plain list. Every baseline's .fit()/evaluate() call then iterates this
    list directly instead of re-triggering per-molecule HDF5 reads from scratch
    on every single pass -- with 5 fits + 8 evaluates sharing train/test data,
    that turns ~13 full HDF5 passes into 2.
    """
    loader = DataLoader(dataset, batch_size=1, collate_fn=_first, num_workers=num_workers)
    out, t0 = [], time.time()
    for i, batch in enumerate(loader):
        out.append(batch)
        print(f"\r  materializing {name}: {i+1}/{len(dataset)}  ({time.time()-t0:.0f}s elapsed)",
                end="", flush=True)
    print()
    return out

eval_buckets = sorted(
    random.Random(BUCKET_SAMPLE_SEED).sample(DEFAULT_BUCKETS, min(N_BUCKETS, len(DEFAULT_BUCKETS)))
)

batcher = Batcher(
    hdf5_db        = HDF5_PATH,
    enc            = enc,
    batches        = eval_buckets,
    seed           = 42,
    atom_size_ceil = 6046,
)
_, _, test_set = batcher.get_sets()

test_loader = _materialize(test_set, "test set")
print(f"Test batches: {len(test_loader)}")

In [ ]:
per_q_stats = {}   # name -> list of per-q-point R² (populated as a side effect of evaluate();
                    # only holds entries for baselines actually run in *this* session --
                    # a resumed/skipped baseline has no per-q breakdown available)

def evaluate(baseline, loader, name):
    """Evaluate a baseline on loader. Returns (mse, r2, us_per_atom).

    Accumulates sum-of-squares globally across the whole test set (not
    per-batch) before computing MSE/R² once at the end. Buckets vary hugely
    in molecule count and target variance (a handful of huge molecules vs.
    thousands of tiny ones), so averaging a per-batch R² across buckets lets
    one low-variance bucket (small ss_tot) swing the whole score wildly --
    this instead matches the standard, statistically stable R² definition.

    Also accumulates the same sums per-q-point (in the same single pass, no
    extra cost) so per_q_stats[name] can show whether a baseline's R² comes
    from genuinely fitting curve shape or just from the easy low-q scale
    signal (I(0) trivially correlates with atom count) collapsing at high q.
    """
    sum_sq_err  = 0.0
    sum_y       = 0.0
    sum_y2      = 0.0
    n_points    = 0
    tpa_weighted = 0.0
    total_atoms  = 0

    Q = len(q_grid)
    sum_sq_err_q = torch.zeros(Q)
    sum_y_q      = torch.zeros(Q)
    sum_y2_q     = torch.zeros(Q)
    n_mols       = 0

    for batch in loader:
        pred, tpa = baseline.timed_call(batch)
        target = torch.log1p(batch.iqval)
        log_pred = torch.log1p(pred.clamp(min=0))
        sq_err = (log_pred - target) ** 2

        sum_sq_err += sq_err.sum().item()
        sum_y      += target.sum().item()
        sum_y2     += (target ** 2).sum().item()
        n_points   += target.numel()

        sum_sq_err_q += sq_err.sum(dim=0).cpu()
        sum_y_q      += target.sum(dim=0).cpu()
        sum_y2_q     += (target ** 2).sum(dim=0).cpu()
        n_mols       += target.shape[0]

        n_atoms = int(batch.padding_mask().sum().item())
        tpa_weighted += tpa * n_atoms
        total_atoms  += n_atoms

    mse = sum_sq_err / n_points if n_points > 0 else float('nan')
    ss_tot = sum_y2 - (sum_y ** 2) / n_points if n_points > 0 else 0.0
    r2 = 1 - sum_sq_err / ss_tot if ss_tot > 0 else float('nan')
    tpa_us = (tpa_weighted / total_atoms * 1e6) if total_atoms > 0 else float('nan')

    if n_mols > 0:
        ss_tot_q = sum_y2_q - (sum_y_q ** 2) / n_mols
        r2_q = torch.where(ss_tot_q > 0, 1 - sum_sq_err_q / ss_tot_q, torch.full_like(ss_tot_q, float('nan')))
        per_q_stats[name] = r2_q.tolist()

    print(f"{name:<30s}  MSE={mse:.4f}  R²={r2:.4f}  {tpa_us:.2f} μs/atom")
    return mse, r2, tpa_us

In [ ]:
sys.path.insert(0, f"{REPO}/Baselines/physics-benchmarks")
sys.path.insert(0, f"{REPO}/Baselines/learned-benchmarks")

from rg import RgBaseline, GuinierPorodBaseline
from atom_count import AtomCountBaseline
from composition_regression import CompositionRegressionBaseline
from pair_peak import BinnedDebyeBaseline

print("=== Physics Baselines ===")
results = load_checkpoint()

# fit on train set where needed -- materialized once, reused by every .fit() below
# and by the learned baselines further down, instead of re-reading HDF5 per call
train_set, _, _ = batcher.get_sets()
train_loader = _materialize(train_set, "train set")

# factories, not baseline instances: construction/fit is deferred until we know
# a baseline isn't already in `results`, so a completed baseline's .fit() never
# re-runs on resume (also fixes the earlier issue where the list literal ran
# every .fit() eagerly before any evaluate() call could print)
physics_baselines = [
    ("Guinier (Rg)",          lambda: RgBaseline(q_grid, energy)),
    ("Guinier-Porod",         lambda: GuinierPorodBaseline(q_grid, energy)),
    ("Atom Count",            lambda: AtomCountBaseline().fit(train_loader)),
    ("Composition Regression",lambda: CompositionRegressionBaseline().fit(train_loader)),
    ("Binned Debye",          lambda: BinnedDebyeBaseline(q_grid, energy)),
]

for name, make_baseline in physics_baselines:
    if name in results:
        print(f"{name:<30s}  (skipped, resumed from checkpoint)")
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
import torch
import torch.nn as nn
from Preprocess import VOCAB
from Baselines.baseline import Baseline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


class TorchMlp(Baseline):
    """GPU-accelerated MLP (replaces sklearn Mlp; targets Kaggle P100/T4)."""

    def __init__(self, hidden=(64, 64), lr=1e-3, epochs=200, mini_batch=256, grad_clip=1.0):
        self.hidden     = hidden
        self.lr         = lr
        self.epochs     = epochs
        self.mini_batch = mini_batch
        self.grad_clip  = grad_clip
        self._net    = None
        self._x_mean = None
        self._x_std  = None

    def _features(self, batch):
        N, M = batch.vocab.shape
        V = len(VOCAB) + 1
        counts = torch.zeros(N, V).scatter_add_(1, batch.vocab.long(), torch.ones(N, M))
        counts[:, 0] = 0.0
        n_atoms = counts.sum(dim=1, keepdim=True).clamp(min=1)
        mask = batch.padding_mask().float()
        r2   = (batch.coord ** 2).sum(dim=-1)
        rg   = ((r2 * mask).sum(dim=1, keepdim=True) / n_atoms).clamp(min=0).sqrt()
        return torch.cat([counts / n_atoms, n_atoms, rg], dim=1).cpu()

    def fit(self, loader):
        X_parts, Y_parts = [], []
        for batch in loader:
            X_parts.append(self._features(batch))
            Y_parts.append(torch.log1p(batch.iqval).cpu())
        X = torch.cat(X_parts)
        Y = torch.cat(Y_parts)
        self._x_mean = X.mean(0)
        self._x_std  = X.std(0).clamp(min=1e-8)
        X = (X - self._x_mean) / self._x_std
        in_dim, out_dim = X.shape[1], Y.shape[1]
        layers, prev = [], in_dim
        for h in self.hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self._net = nn.Sequential(*layers).to(DEVICE)
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        opt  = torch.optim.Adam(self._net.parameters(), lr=self.lr)
        n    = len(X)
        self._net.train()
        for _ in range(self.epochs):
            perm = torch.randperm(n, device=DEVICE)
            for i in range(0, n, self.mini_batch):
                b    = perm[i:i + self.mini_batch]
                loss = nn.functional.mse_loss(self._net(X[b]), Y[b])
                opt.zero_grad()
                loss.backward()
                # heavy-tailed features (atom counts / Rg span ~1 to 6046 atoms) can
                # produce occasional large gradients that blow Adam up to inf/nan
                # over 200 epochs; clip so one bad minibatch can't diverge the run.
                nn.utils.clip_grad_norm_(self._net.parameters(), self.grad_clip)
                opt.step()
        self._net.eval()
        return self

    def __call__(self, batch):
        X = (self._features(batch) - self._x_mean) / self._x_std
        with torch.no_grad():
            log_pred = self._net(X.to(DEVICE))
        # safety net: clamp before expm1 so a NaN/exploded weight (despite grad
        # clipping) surfaces as a large finite MSE instead of inf, which would
        # otherwise break evaluate()'s accumulated sums and the summary plot.
        log_pred = torch.nan_to_num(log_pred, nan=0.0, posinf=30.0, neginf=0.0).clamp(max=30.0)
        return torch.expm1(log_pred).cpu()

    def timed_call(self, batch):
        X = (self._features(batch) - self._x_mean) / self._x_std
        X = X.to(DEVICE)
        if DEVICE.type == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end   = torch.cuda.Event(enable_timing=True)
            start.record()
            with torch.no_grad():
                log_pred = self._net(X)
            end.record()
            torch.cuda.synchronize()
            elapsed = start.elapsed_time(end) / 1000.0
        else:
            import time
            t0 = time.process_time()
            with torch.no_grad():
                log_pred = self._net(X)
            elapsed = time.process_time() - t0
        log_pred = torch.nan_to_num(log_pred, nan=0.0, posinf=30.0, neginf=0.0).clamp(max=30.0)
        pred    = torch.expm1(log_pred).cpu()
        n_atoms = int(batch.padding_mask().sum().item())
        return pred, elapsed / max(n_atoms, 1)


In [ ]:
from linsvm import Linsvm
from nearest_neighbour import NNBaseline

print("=== Learned Baselines ===")

learned_baselines = [
    ("MLP",                lambda: TorchMlp().fit(train_loader)),
    ("Linear SVM",         lambda: Linsvm().fit(train_loader)),
    ("Nearest Neighbour",  lambda: NNBaseline(q_grid, energy).fit(train_loader)),
]

for name, make_baseline in learned_baselines:
    if name in results:
        print(f"{name:<30s}  (skipped, resumed from checkpoint)")
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
print("\n=== Summary ===")
print(f"{'Baseline':<30s}  {'MSE':>8s}  {'R²':>8s}  {'μs/atom':>10s}")
print("-" * 62)
for name, (mse, r2, tpa) in sorted(results.items(), key=lambda x: x[1][0]):
    print(f"{name:<30s}  {mse:>8.4f}  {r2:>8.4f}  {tpa:>10.2f}")

In [ ]:
import math
import matplotlib.pyplot as plt

# fixed categorical order (identity per baseline), never reassigned by rank
PALETTE = [
    "#2a78d6", "#1baf7a", "#eda100", "#008300",
    "#4a3aa7", "#e34948", "#e87ba4", "#eb6834"
]
TEXT_PRIMARY, TEXT_MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

ordered = sorted(results.items(), key=lambda x: x[1][0])  # ranked by MSE, best first
names   = [n for n, _ in ordered]
colors  = [PALETTE[i % len(PALETTE)] for i in range(len(ordered))]
mse_v   = [v[0] for _, v in ordered]
r2_v    = [v[1] for _, v in ordered]
tpa_v   = [v[2] for _, v in ordered]

n = len(ordered)
fig, axes = plt.subplots(1, 3, figsize=(16, 0.85 * n + 1.8))
fig.subplots_adjust(wspace=0.45)
specs = [
    (axes[0], mse_v, "MSE (log1p I)", "{:.4f}"),
    (axes[1], r2_v,  "R²",            "{:.3f}"),
    (axes[2], tpa_v, "μs / atom",     "{:.1f}"),
]

y = range(n)
for ax, vals, title, fmt in specs:
    # NaN/Inf are possible (e.g. R² is NaN when a molecule's ss_tot == 0, or an
    # under-trained baseline predicts values that blow up log1p) -- draw those
    # bars as zero-width with an "n/a" label instead of feeding non-finite
    # numbers into matplotlib, which raises on set_xlim.
    finite_vals = [v for v in vals if math.isfinite(v)]
    xmax = max(finite_vals) if finite_vals else 1
    plot_vals = [v if math.isfinite(v) else 0 for v in vals]

    bars = ax.barh(y, plot_vals, color=colors, height=0.68, zorder=3)
    ax.set_yticks(list(y))
    ax.set_yticklabels(names if ax is axes[0] else [], color=TEXT_PRIMARY, fontsize=10)
    ax.set_ylim(n - 0.5, -0.5)  # best (lowest MSE) at top, matches sort order
    ax.set_title(title, color=TEXT_PRIMARY, fontsize=12, pad=10)
    ax.tick_params(colors=TEXT_MUTED, length=0, labelsize=9)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(axis="x", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for bar, v in zip(bars, vals):
        label = "n/a" if not math.isfinite(v) else fmt.format(v)
        ax.text(bar.get_width() + 0.02 * xmax, bar.get_y() + bar.get_height() / 2,
                label, va="center", ha="left", fontsize=9, color=TEXT_PRIMARY)
    ax.set_xlim(0, xmax * 1.22 if xmax > 0 else 1)

fig.suptitle("Baseline comparison (sorted by MSE, best first)", color=TEXT_PRIMARY, fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("output.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# ── R²(q): does performance hold up at high q, or is it riding the easy low-q
# scale signal? ────────────────────────────────────────────────────────────
# The pooled R² above is dominated by molecule-to-molecule scale variance --
# I(0) roughly tracks atom count, so a baseline that captures nothing but size
# already explains most of the pooled variance (see Atom Count's R²). This
# plot breaks R² out per q-point instead: low q is where that easy scale
# signal lives, high q is where fine structural shape (pair correlations,
# Porod tail) actually has to be predicted. A baseline whose line collapses
# at high q is winning on scale alone, not on physics.
q_np = q_grid.numpy()
available = [(n, r2q) for n, r2q in per_q_stats.items() if n in results]
missing = [n for n in results if n not in per_q_stats]
if missing:
    print(f"(no per-q breakdown for {missing} -- skipped via checkpoint resume this session)")

if available:
    fig2, ax2 = plt.subplots(figsize=(11, 6))
    name_to_color = {n: PALETTE[i % len(PALETTE)] for i, (n, _) in enumerate(ordered)}

    for name, r2q in available:
        ax2.plot(q_np, r2q, color=name_to_color.get(name, TEXT_MUTED), linewidth=2, label=name)

    ax2.axhline(0, color=TEXT_MUTED, linewidth=1, linestyle="--", zorder=1)
    ax2.set_xlabel("q (Å⁻¹)", color=TEXT_PRIMARY)
    ax2.set_ylabel("R² at this q-point", color=TEXT_PRIMARY)
    ax2.set_title("Per-q R²: low q (scale) vs. high q (structural shape)", color=TEXT_PRIMARY, fontsize=12)
    ax2.tick_params(colors=TEXT_MUTED)
    for spine in ax2.spines.values():
        spine.set_visible(False)
    ax2.grid(color=GRID, linewidth=0.8, zorder=0)
    ax2.set_axisbelow(True)
    # floor the view at -3: a single badly-diverging baseline can otherwise
    # squash the whole readable range for everyone else
    finite_r2q = [v for _, r2q in available for v in r2q if math.isfinite(v)]
    floor = max(-3.0, min(finite_r2q)) if finite_r2q else -3.0
    ax2.set_ylim(floor - 0.1, 1.05)
    ax2.legend(loc="lower left", frameon=False, labelcolor=TEXT_PRIMARY, fontsize=9)

    fig2.tight_layout()
    fig2.savefig("output_per_q_r2.png", dpi=600, bbox_inches="tight")
    plt.show()